In [ ]:
# ============= z200 vs z500 cut-off co-location: z500 So distribution =============
# INDEPENDENT cell. All AMJJA z200 closed cut-offs (ex==1) are matched to the nearest
# SAME-DAY z500 closed cut-off (ex==1) within a horizontal radius; we then plot the
# z500 So distribution of the matched z500 lows vs. the full z500 cut-off population.
import os, numpy as np, pandas as pd, xarray as xr, matplotlib.pyplot as plt

_PROJ = "${ANALOGUE_BASE}"
CATALOGUE_NC_500 = os.path.join(_PROJ, "col_catalogue_AMJJA",        "col_features_AMJJA_1940_2025_largebox.nc")
CATALOGUE_NC_200 = os.path.join(_PROJ, "col_catalogue_AMJJA_z200", "col_features_AMJJA_1940_2025_largebox_z200.nc")

MATCH_RADIUS_KM    = None    # None -> use each z500 feature's own So-optimal radius R; set a number for a fixed km tolerance
RADIUS_FACTOR      = 1.0     # scale on R when MATCH_RADIUS_KM is None
FALLBACK_RADIUS_KM = 800.0   # used only if the catalogue has no radius variable
SO_MIN_200 = 0.0             # keep ALL ex==1 z200 cut-offs
SO_MIN_500 = 0.0             # keep ALL ex==1 z500 cut-offs

def _pick(ds, *cands):
    for c in cands:
        if c in ds.variables:
            return c
    return None

def load_catalogue(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"catalogue not found (200 hPa run merged yet?): {path}")
    ds = xr.open_dataset(path)
    v = dict(time=_pick(ds, "time", "date", "valid_time"),
             lon =_pick(ds, "lon", "longitude", "clon", "lon_center", "x"),
             lat =_pick(ds, "lat", "latitude", "clat", "lat_center", "y"),
             So  =_pick(ds, "So", "so", "SO"),
             ex  =_pick(ds, "ex", "EX", "closed"),
             R   =_pick(ds, "R", "radius", "Ropt", "r_opt", "R_opt", "radius_km", "Rso"))
    if None in (v["time"], v["lon"], v["lat"], v["So"], v["ex"]):
        raise KeyError(f"missing a required variable in {os.path.basename(path)}; "
                       f"found {list(ds.variables)} -> mapped {v}")
    cols = [v["time"], v["lon"], v["lat"], v["So"], v["ex"]] + ([v["R"]] if v["R"] else [])
    df = ds[cols].to_dataframe().reset_index()
    ds.close()
    ren = {v["time"]: "time", v["lon"]: "lon", v["lat"]: "lat", v["So"]: "So", v["ex"]: "ex"}
    if v["R"]:
        ren[v["R"]] = "R"
    df = df.rename(columns=ren)
    if "R" not in df.columns:
        df["R"] = np.nan
    df["day"] = pd.DatetimeIndex(df["time"]).normalize()
    return df[["day", "lon", "lat", "So", "ex", "R"]], v

df200_all, map200 = load_catalogue(CATALOGUE_NC_200)
df500_all, map500 = load_catalogue(CATALOGUE_NC_500)
print("200 hPa var-map:", map200)
print("500 hPa var-map:", map500)

c200 = df200_all[(df200_all.ex == 1) & (df200_all.So >= SO_MIN_200)].copy()
c500 = df500_all[(df500_all.ex == 1) & (df500_all.So >= SO_MIN_500)].copy()
print(f"ex==1 cut-offs:  z200 = {len(c200)}   z500 = {len(c500)}")

_RE = 6371.0
def _hav(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin(np.radians(lat2 - lat1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * _RE * np.arcsin(np.sqrt(a))

c500_by_day = {d: g.reset_index(drop=True) for d, g in c500.groupby("day")}
rows = []
for d, g2 in c200.groupby("day"):
    g5 = c500_by_day.get(d)
    if g5 is None:
        continue
    lat5, lon5 = g5["lat"].to_numpy(), g5["lon"].to_numpy()
    So5, R5 = g5["So"].to_numpy(), g5["R"].to_numpy()
    for i2, r2 in g2.iterrows():
        dist = _hav(r2["lat"], r2["lon"], lat5, lon5)
        j = int(np.argmin(dist)); dmin = float(dist[j])
        tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
              (R5[j] * RADIUS_FACTOR if np.isfinite(R5[j]) else FALLBACK_RADIUS_KM)
        if dmin <= tol:
            rows.append((d, i2, j, dmin, So5[j], R5[j]))

mp = pd.DataFrame(rows, columns=["day", "i200", "j500", "dist_km", "So500", "R500"])
matched_500 = mp.drop_duplicates(subset=["day", "j500"])       # a z500 low may partner several z200 lows
So500_matched = matched_500["So500"].to_numpy()
So500_all = c500["So"].to_numpy()

print(f"z200 cut-offs with a z500 partner: {mp['i200'].nunique()} / {len(c200)} "
      f"({(mp['i200'].nunique() / max(len(c200),1)):.1%})")
print(f"unique matched z500 cut-offs:      {len(matched_500)} / {len(c500)} "
      f"({(len(matched_500) / max(len(c500),1)):.1%})")
print(f"z500 So  median  matched={np.median(So500_matched):.2f}  all={np.median(So500_all):.2f} | "
      f"mean matched={np.mean(So500_matched):.2f} all={np.mean(So500_all):.2f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.arange(0, np.nanpercentile(So500_all, 99.5) + 2, 1.0)
ax.hist(So500_all, bins=bins, density=False, histtype="stepfilled", alpha=0.35,
        color="0.55", label=f"all z500 cut-offs  (N={len(So500_all)})")
ax.hist(So500_matched, bins=bins, density=False, histtype="step", lw=2.2, color="#1f6fb2",
        label=f"z500 with z200 partner  (N={len(So500_matched)})")
ax.axvline(np.median(So500_all), color="0.4", ls="--", lw=1.1)
ax.axvline(np.median(So500_matched), color="#1f6fb2", ls="--", lw=1.1)
for xv in (6, 10):
    ax.axvline(xv, color="k", ls=":", lw=1.0)
    ax.text(xv + 0.1, ax.get_ylim()[1] * 0.9, f"So={xv}", fontsize=8)
ax.set_xlabel("z500 So  (m / 100 km)"); ax.set_ylabel("Total Number of detected Cut-Offs")
ax.set_title("z500 So — cut-offs vertically coherent with a z200 cut-off vs. all")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# ===== z200 perspective: match each z500 cut-off to the nearest same-day z200 cut-off =====
# (mirror of the previous cell). Plot the z200 So distribution of matched z200 lows vs. all.
# Reuses c200, c500, _hav, MATCH_RADIUS_KM, RADIUS_FACTOR, FALLBACK_RADIUS_KM from the cell above.
import numpy as np, pandas as pd, matplotlib.pyplot as plt

c200_by_day = {d: g.reset_index(drop=True) for d, g in c200.groupby("day")}
rows2 = []
for d, g5 in c500.groupby("day"):
    g2 = c200_by_day.get(d)
    if g2 is None:
        continue
    lat2, lon2 = g2["lat"].to_numpy(), g2["lon"].to_numpy()
    So2, R2 = g2["So"].to_numpy(), g2["R"].to_numpy()
    for i5, r5 in g5.iterrows():
        dist = _hav(r5["lat"], r5["lon"], lat2, lon2)
        j = int(np.argmin(dist)); dmin = float(dist[j])
        tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
              (R2[j] * RADIUS_FACTOR if np.isfinite(R2[j]) else FALLBACK_RADIUS_KM)   # target = z200 radius
        if dmin <= tol:
            rows2.append((d, i5, j, dmin, So2[j], R2[j]))

mp2 = pd.DataFrame(rows2, columns=["day", "i500", "j200", "dist_km", "So200", "R200"])
matched_200 = mp2.drop_duplicates(subset=["day", "j200"])      # a z200 low may partner several z500 lows
So200_matched = matched_200["So200"].to_numpy()
So200_all = c200["So"].to_numpy()

print(f"z500 cut-offs with a z200 partner: {mp2['i500'].nunique()} / {len(c500)} "
      f"({(mp2['i500'].nunique() / max(len(c500),1)):.1%})")
print(f"unique matched z200 cut-offs:      {len(matched_200)} / {len(c200)} "
      f"({(len(matched_200) / max(len(c200),1)):.1%})")
print(f"z200 So  median  matched={np.median(So200_matched):.2f}  all={np.median(So200_all):.2f} | "
      f"mean matched={np.mean(So200_matched):.2f} all={np.mean(So200_all):.2f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.arange(0, np.nanpercentile(So200_all, 99.5) + 2, 1.0)
ax.hist(So200_all, bins=bins, density=False, histtype="stepfilled", alpha=0.35,
        color="0.55", label=f"all z200 cut-offs  (N={len(So200_all)})")
ax.hist(So200_matched, bins=bins, density=False, histtype="step", lw=2.2, color="#b2461f",
        label=f"z200 with z500 partner  (N={len(So200_matched)})")
ax.axvline(np.median(So200_all), color="0.4", ls="--", lw=1.1)
ax.axvline(np.median(So200_matched), color="#b2461f", ls="--", lw=1.1)
for xv in (6, 10):
    ax.axvline(xv, color="k", ls=":", lw=1.0)
    ax.text(xv + 0.1, ax.get_ylim()[1] * 0.9, f"So={xv}", fontsize=8)
ax.set_xlabel("z200 So  (m / 100 km)"); ax.set_ylabel("Total Number of detected Cut-Offs")
ax.set_title("z200 So — cut-offs vertically coherent with a z500 cut-off vs. all")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# ===== Co-location numbers for the study qualifiers: z500 So>=6  &  z200 So>=10 =====
# Reuses df500_all, df200_all, _hav, MATCH_RADIUS_KM, RADIUS_FACTOR, FALLBACK_RADIUS_KM.
import numpy as np, pandas as pd
SO_500_Q, SO_200_Q = 6.0, 10.0
c500q = df500_all[(df500_all.ex == 1) & (df500_all.So >= SO_500_Q)].copy()
c200q = df200_all[(df200_all.ex == 1) & (df200_all.So >= SO_200_Q)].copy()
def match(src, tgt):        
    tgt_by = {d: g.reset_index(drop=True) for d, g in tgt.groupby("day")}
    rows = []
    for d, gs in src.groupby("day"):
        gt = tgt_by.get(d)
        if gt is None:
            continue
        latt, lont, Sot, Rt = gt["lat"].to_numpy(), gt["lon"].to_numpy(), gt["So"].to_numpy(), gt["R"].to_numpy()
        for i, r in gs.iterrows():
            dist = _hav(r["lat"], r["lon"], latt, lont); j = int(np.argmin(dist)); dmin = float(dist[j])
            tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
                  (Rt[j] * RADIUS_FACTOR if np.isfinite(Rt[j]) else FALLBACK_RADIUS_KM)
            if dmin <= tol:
                rows.append((d, i, j, dmin, Sot[j]))
    return pd.DataFrame(rows, columns=["day", "isrc", "jtgt", "dist_km", "So_tgt"])

m_5to2 = match(c500q, c200q)              # z500(So>=6) -> nearest z200(So>=10)
m_2to5 = match(c200q, c500q)              # z200(So>=10) -> nearest z500(So>=6)
u200 = m_5to2.drop_duplicates(["day", "jtgt"])["So_tgt"].to_numpy()   # z200 So of the z200 partners
u500 = m_2to5.drop_duplicates(["day", "jtgt"])["So_tgt"].to_numpy()   # z500 So of the z500 partners

print(f"qualifiers: z500 So>={SO_500_Q:.0f} (N={len(c500q)})   z200 So>={SO_200_Q:.0f} (N={len(c200q)})")
print(f"z500(So>=6)  with a z200(So>=10) partner: {m_5to2['isrc'].nunique():5d} / {len(c500q)}  "
      f"({m_5to2['isrc'].nunique()/max(len(c500q),1):.1%})")
print(f"z200(So>=10) with a z500(So>=6)  partner: {m_2to5['isrc'].nunique():5d} / {len(c200q)}  "
      f"({m_2to5['isrc'].nunique()/max(len(c200q),1):.1%})")
if len(u200):
    print(f"matched z200 So (partners of z500>=6) : median {np.median(u200):.2f}  mean {u200.mean():.2f}  (N={len(u200)})")
if len(u500):
    print(f"matched z500 So (partners of z200>=10): median {np.median(u500):.2f}  mean {u500.mean():.2f}  (N={len(u500)})")

In [ ]:
# ===== Vertical-structure of z200(>=10) vs 500 hPa, as a function of a FIXED match radius =====
# closed = ex1 So>=6 ; trough = ex0 So>=3 ; classify by nearest within a fixed km radius.
import numpy as np, pandas as pd
c200q       = df200_all[(df200_all.ex == 1) & (df200_all.So >= 10.0)].copy()
c500_closed = df500_all[(df500_all.ex == 1) & (df500_all.So >= 3.0)].copy()
c500_trough = df500_all[(df500_all.ex == 0) & (df500_all.So >= 3.0)].copy()

def within(src, tgt, radius_km):                 # src cut-offs that have ANY tgt within radius_km, same day
    tgt_by = {d: (g["lat"].to_numpy(), g["lon"].to_numpy()) for d, g in tgt.groupby("day")}
    hit = set()
    for d, gs in src.groupby("day"):
        t = tgt_by.get(d)
        if t is None:
            continue
        latt, lont = t
        for i, r in gs.iterrows():
            if _hav(r["lat"], r["lon"], latt, lont).min() <= radius_km:
                hit.add(i)
    return hit

n = len(c200q)
print(f"z200(So>=10) N={n}   |   fixed-radius vertical-structure sweep")
print(f"{'radius_km':>9} {'closed%':>8} {'trough%':>8} {'neither%':>9}")
for R in (300, 500, 750, 1000, 1500, 2000, 2500, 3000):
    wc = within(c200q, c500_closed, R)
    wt = within(c200q[~c200q.index.isin(wc)], c500_trough, R)
    nc, nt = len(wc), len(wt); nn = n - nc - nt
    print(f"{R:9d} {100*nc/n:8.1f} {100*nt/n:8.1f} {100*nn/n:9.1f}")

In [ ]:
# ===== z500 So distribution for z500 cut-offs matched to a z200 cut-off with So200 in [10, 11] =====
# Reuses df200_all, df500_all, _hav from the previous matching cell (falls back to reload if absent).
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SO200_LO, SO200_HI = 10.0, 11.0     # z200 So band (Kasuga-threshold neighbourhood)
MATCH_RADIUS_KM    = None           # None -> use each z500 feature's own So-optimal radius R
RADIUS_FACTOR      = 1.0
FALLBACK_RADIUS_KM = 800.0

# z200 restricted to the So band; z500 = all closed cut-offs in the catalogue (So >= 3 floor)
c200 = df200_all[(df200_all.ex == 1) & (df200_all.So >= SO200_LO) & (df200_all.So <= SO200_HI)].copy()
c500 = df500_all[df500_all.ex == 1].copy()
print(f"z200 cut-offs with So in [{SO200_LO:.0f},{SO200_HI:.0f}]: {len(c200)}   |   z500 cut-offs (all): {len(c500)}")

c500_by_day = {d: g.reset_index(drop=True) for d, g in c500.groupby("day")}
rows = []
for d, g2 in c200.groupby("day"):
    g5 = c500_by_day.get(d)
    if g5 is None:
        continue
    lat5, lon5 = g5["lat"].to_numpy(), g5["lon"].to_numpy()
    So5, R5 = g5["So"].to_numpy(), g5["R"].to_numpy()
    for _, r2 in g2.iterrows():
        dist = _hav(r2["lat"], r2["lon"], lat5, lon5)
        j = int(np.argmin(dist)); dmin = float(dist[j])
        tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
              (R5[j] * RADIUS_FACTOR if np.isfinite(R5[j]) else FALLBACK_RADIUS_KM)
        if dmin <= tol:
            rows.append((d, j, So5[j]))

mp = pd.DataFrame(rows, columns=["day", "j500", "So500"])
matched = mp.drop_duplicates(subset=["day", "j500"])          # count each z500 low once
So500 = matched["So500"].to_numpy()
print(f"matched z500 cut-offs (unique): {len(matched)}  "
      f"| z200 lows with a z500 partner: {mp.shape[0]} / {len(c200)} ({mp.shape[0]/max(len(c200),1):.1%})")
if len(So500):
    qs = np.percentile(So500, [10, 25, 50, 75, 90])
    print(f"z500 So  mean={So500.mean():.2f}  median={np.median(So500):.2f}  "
          f"| p10/25/50/75/90 = {qs.round(2)}")
    print(f"z500 So >= 6: {(So500>=6).mean():.1%}   >= 10: {(So500>=10).mean():.1%}")

fig, ax = plt.subplots(figsize=(7, 4.5))
bins = np.arange(0, (np.nanpercentile(So500, 99.5) if len(So500) else 20) + 2, 1.0)
ax.hist(So500, bins=bins, color="#1f6fb2", alpha=0.8, edgecolor="w")
if len(So500):
    ax.axvline(np.median(So500), color="k", ls="--", lw=1.3,
               label=f"median z500 So = {np.median(So500):.1f}")
for xv in (6, 10):
    ax.axvline(xv, color="0.35", ls=":", lw=1.0)
    ax.text(xv + 0.1, ax.get_ylim()[1] * 0.92, f"So={xv}", fontsize=8, color="0.35")
ax.set_xlabel("z500 So  (m / 100 km)"); ax.set_ylabel("count")
ax.set_title(f"z500 So of cut-offs under a z200 cut-off with So200 in [{SO200_LO:.0f}, {SO200_HI:.0f}]\n"
             f"(N = {len(So500)} matched z500 lows)")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# ===== per-pair So200/So500 scaling ratio, with bootstrap CI on the median =====
# Each z200 closed cut-off (ex==1) is matched to the nearest same-day z500 closed cut-off
# within radius; ratio = So200 / So500 per matched pair. Reuses df200_all, df500_all, _hav.
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SO200_LO           = 10.0      # z200 So range to include (3 = whole catalogue; set 10 for Kasuga-qualifying only)
MATCH_RADIUS_KM    = None     # None -> each z500 feature's own So-optimal radius R
RADIUS_FACTOR      = 1.0
FALLBACK_RADIUS_KM = 800.0
N_BOOT             = 10000
rng = np.random.default_rng(0)

c200 = df200_all[(df200_all.ex == 1) & (df200_all.So >= SO200_LO)].copy()
c500 = df500_all[df500_all.ex == 1].copy()
c500_by_day = {d: g.reset_index(drop=True) for d, g in c500.groupby("day")}

rows = []
for d, g2 in c200.groupby("day"):
    g5 = c500_by_day.get(d)
    if g5 is None:
        continue
    lat5, lon5 = g5["lat"].to_numpy(), g5["lon"].to_numpy()
    So5, R5 = g5["So"].to_numpy(), g5["R"].to_numpy()
    for _, r2 in g2.iterrows():
        dist = _hav(r2["lat"], r2["lon"], lat5, lon5)
        j = int(np.argmin(dist)); dmin = float(dist[j])
        tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
              (R5[j] * RADIUS_FACTOR if np.isfinite(R5[j]) else FALLBACK_RADIUS_KM)
        if dmin <= tol and So5[j] > 0:
            rows.append((r2["So"], So5[j], r2["So"] / So5[j]))

pairs = pd.DataFrame(rows, columns=["So200", "So500", "ratio"])
ratio = pairs["ratio"].to_numpy()
med = np.median(ratio)
boot = np.median(rng.choice(ratio, size=(N_BOOT, len(ratio)), replace=True), axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f"matched pairs (z200 So>={SO200_LO:.0f}): N = {len(pairs)}")
print(f"per-pair So200/So500 ratio:  median = {med:.3f}   95% bootstrap CI = [{lo:.3f}, {hi:.3f}]")
print(f"                              mean = {ratio.mean():.3f}   IQR = "
      f"[{np.percentile(ratio,25):.3f}, {np.percentile(ratio,75):.3f}]")
print(f"ratio-of-medians (for comparison): {np.median(pairs.So200)/np.median(pairs.So500):.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(ratio, bins=np.arange(0, np.nanpercentile(ratio, 99) + 0.2, 0.1),
        color="#8e6fb2", alpha=0.8, edgecolor="w")
ax.axvline(med, color="k", ls="--", lw=1.4, label=f"median = {med:.2f}  (95% CI {lo:.2f}-{hi:.2f})")
ax.axvspan(lo, hi, color="k", alpha=0.10)
ax.set_xlabel("So200 / So500  (per matched pair)"); ax.set_ylabel("count")
ax.set_title(f"200/500 hPa So scaling ratio  (N = {len(pairs)} matched cut-off pairs)")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# ===== bootstrap CI on the conditional median So500 given z200 So in [10,11] =====
# The direct threshold statistic: what 500 hPa strength sits under a Kasuga-qualifying
# (So200 ~ 10) upper cut-off? Reuses df200_all, df500_all, _hav.
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SO200_LO, SO200_HI = 10.0, 11.0
MATCH_RADIUS_KM, RADIUS_FACTOR, FALLBACK_RADIUS_KM = None, 1.0, 800.0
N_BOOT = 10000
rng = np.random.default_rng(0)

c200 = df200_all[(df200_all.ex == 1) & (df200_all.So >= SO200_LO) & (df200_all.So <= SO200_HI)].copy()
c500 = df500_all[df500_all.ex == 1].copy()
c500_by_day = {d: g.reset_index(drop=True) for d, g in c500.groupby("day")}

rows = []
for d, g2 in c200.groupby("day"):
    g5 = c500_by_day.get(d)
    if g5 is None:
        continue
    lat5, lon5, So5, R5 = g5["lat"].to_numpy(), g5["lon"].to_numpy(), g5["So"].to_numpy(), g5["R"].to_numpy()
    for _, r2 in g2.iterrows():
        dist = _hav(r2["lat"], r2["lon"], lat5, lon5)
        j = int(np.argmin(dist)); dmin = float(dist[j])
        tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
              (R5[j] * RADIUS_FACTOR if np.isfinite(R5[j]) else FALLBACK_RADIUS_KM)
        if dmin <= tol:
            rows.append((d, j, So5[j]))

matched = pd.DataFrame(rows, columns=["day", "j500", "So500"]).drop_duplicates(subset=["day", "j500"])
So500 = matched["So500"].to_numpy()

med = np.median(So500)
boot = np.median(rng.choice(So500, size=(N_BOOT, len(So500)), replace=True), axis=1)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"z200 So in [{SO200_LO:.0f},{SO200_HI:.0f}] -> matched unique z500 cut-offs: N = {len(So500)}")
print(f"conditional median So500 = {med:.2f}   95% bootstrap CI = [{lo:.2f}, {hi:.2f}]")
print(f"mean So500 = {So500.mean():.2f} | share So500>=6: {(So500>=6).mean():.1%}")

fig, ax = plt.subplots(figsize=(7, 4.3))
ax.hist(So500, bins=np.arange(0, np.nanpercentile(So500, 99.5) + 2, 1.0),
        color="#1f6fb2", alpha=0.8, edgecolor="w")
ax.axvline(med, color="k", ls="--", lw=1.4, label=f"median So500 = {med:.1f}  (95% CI {lo:.1f}-{hi:.1f})")
ax.axvspan(lo, hi, color="k", alpha=0.10)
ax.axvline(6, color="0.35", ls=":", lw=1.0); ax.text(6.1, ax.get_ylim()[1]*0.9, "So=6", fontsize=8, color="0.35")
ax.set_xlabel("z500 So  (m / 100 km)"); ax.set_ylabel("count")
ax.set_title(f"z500 So under a Kasuga-threshold z200 cut-off (So200 in [{SO200_LO:.0f},{SO200_HI:.0f}], N={len(So500)})")
ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
# ===== 2x2: (a) z200 So matched vs all | (b) conditional So500 | (c) So200/So500 ratio | (d) daily activity =====
# Reuses df200_all, df500_all, _hav.
import numpy as np, pandas as pd, matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 14, "axes.titlesize": 15, "axes.labelsize": 14,
                     "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11})
MATCH_RADIUS_KM, RADIUS_FACTOR, FALLBACK_RADIUS_KM = None, 1.0, 800.0
N_BOOT = 10000
rng = np.random.default_rng(0)

def nearest_pairs(src, tgt):
    tby = {d: g.reset_index(drop=True) for d, g in tgt.groupby("day")}
    out = []
    for d, gs in src.groupby("day"):
        gt = tby.get(d)
        if gt is None:
            continue
        latt, lont, Sot, Rt = gt["lat"].to_numpy(), gt["lon"].to_numpy(), gt["So"].to_numpy(), gt["R"].to_numpy()
        for i, r in gs.iterrows():
            dist = _hav(r["lat"], r["lon"], latt, lont); j = int(np.argmin(dist)); dmin = float(dist[j])
            tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
                  (Rt[j] * RADIUS_FACTOR if np.isfinite(Rt[j]) else FALLBACK_RADIUS_KM)
            if dmin <= tol and Sot[j] > 0:
                out.append((d, i, j, dmin, Sot[j], r["So"]))
    return pd.DataFrame(out, columns=["day", "isrc", "jtgt", "dist_km", "So_tgt", "So_src"])

def _letter(ax, s):
    ax.text(0.02, 0.97, f"({s})", transform=ax.transAxes, fontsize=18, fontweight="bold", va="top")

fig, axes = plt.subplots(2, 2, figsize=(16, 11)); axf = axes.ravel()

# ---------- (a) z200 So: all vs matched from z500(any So) vs matched from z500(So>=6) ----------
c200A     = df200_all[df200_all.ex == 1].copy()
c500_any  = df500_all[df500_all.ex == 1].copy()
c500_q6   = df500_all[(df500_all.ex == 1) & (df500_all.So >= 6.0)].copy()
m_any = nearest_pairs(c500_any, c200A).drop_duplicates(["day", "jtgt"])["So_tgt"].to_numpy()
m_q6  = nearest_pairs(c500_q6,  c200A).drop_duplicates(["day", "jtgt"])["So_tgt"].to_numpy()
So200_all = c200A["So"].to_numpy()
ax = axf[0]; bins = np.arange(0, np.nanpercentile(So200_all, 99.5) + 2, 1.0)
ax.hist(So200_all, bins=bins, histtype="stepfilled", alpha=0.35, color="0.55", label=f"all z200 (N={len(So200_all)})")
ax.hist(m_any, bins=bins, histtype="step", lw=2.4, color="#F29424", label=f"z500 any So (N={len(m_any)})")
ax.hist(m_q6,  bins=bins, histtype="step", lw=2.4, color="#2B7C8B", label=f"z500 So≥6 (N={len(m_q6)})")
ax.axvline(np.median(So200_all), color="0.4", ls="--", lw=1.0)
ax.axvline(np.median(m_any), color="#F29424", ls="--", lw=1.0)
ax.axvline(np.median(m_q6), color="#2B7C8B", ls="--", lw=1.0)
ax.set_xlabel("z200 So  (m / 100 km)"); ax.set_ylabel("Number of detected Cut-Off Lows")
ax.set_title("z200 So — all vs matched from z500 (any So / So≥6)")
ax.legend(frameon=False, loc="upper right"); _letter(ax, "a")

# ---------- (b) conditional median So500 given z200 So in [9,11] ----------
c200B = df200_all[(df200_all.ex == 1) & (df200_all.So >= 9.0) & (df200_all.So <= 11.0)].copy()
c500B = df500_all[df500_all.ex == 1].copy()
So500 = nearest_pairs(c200B, c500B).drop_duplicates(["day", "jtgt"])["So_tgt"].to_numpy()
medC = np.median(So500); bootC = np.median(rng.choice(So500, size=(N_BOOT, len(So500)), replace=True), axis=1)
loC, hiC = np.percentile(bootC, [2.5, 97.5])
print(f"(b) So500|z200in[9,11]: N={len(So500)}  median={medC:.2f}  95%CI=[{loC:.2f},{hiC:.2f}]  share>=6: {(So500>=6).mean():.1%}")
ax = axf[1]
ax.hist(So500, bins=np.arange(0, np.nanpercentile(So500, 99.5) + 2, 1.0), color="#4FA7BF", alpha=0.8, edgecolor="w")
ax.axvline(medC, color="k", ls="--", lw=1.6, label=f"median So500 = {medC:.1f}  (95% CI {loC:.1f}-{hiC:.1f})")
ax.axvspan(loC, hiC, color="k", alpha=0.10)
ax.axvline(6, color="0.35", ls=":", lw=1.0); ax.text(6.1, ax.get_ylim()[1] * 0.9, "So=6", fontsize=11, color="0.35")
ax.set_xlabel("z500 So  (m / 100 km)"); ax.set_ylabel("Number of detected Cut-Off Lows")
ax.set_title(f"z500 So under z200 cut-off So∈[9,11] (N={len(So500)})")
ax.legend(frameon=False); _letter(ax, "b")

# ---------- (c) per-pair So200/So500 scaling ratio ----------
c200C = df200_all[(df200_all.ex == 1) & (df200_all.So >= 10.0)].copy()
c500C = df500_all[df500_all.ex == 1].copy()
ratio = (lambda p: (p["So_src"] / p["So_tgt"]).to_numpy())(nearest_pairs(c200C, c500C))
medR = np.median(ratio); bootR = np.median(rng.choice(ratio, size=(N_BOOT, len(ratio)), replace=True), axis=1)
loR, hiR = np.percentile(bootR, [2.5, 97.5])
print(f"(c) ratio: N={len(ratio)}  median={medR:.3f}  95%CI=[{loR:.3f},{hiR:.3f}]  mean={ratio.mean():.3f}")
ax = axf[2]
ax.hist(ratio, bins=np.arange(0, np.nanpercentile(ratio, 99) + 0.2, 0.1), color="#F29424", alpha=0.8, edgecolor="w")
ax.axvline(medR, color="k", ls="--", lw=1.6, label=f"median = {medR:.2f}  (95% CI {loR:.2f}-{hiR:.2f})")
ax.axvspan(loR, hiR, color="k", alpha=0.10)
ax.set_xlabel("So200 / So500  (per matched pair)"); ax.set_ylabel("Number of detected Cut-Off Lows")
ax.set_title(f"200/500 hPa So scaling ratio (N={len(ratio)})")
ax.legend(frameon=False); _letter(ax, "c")

# ---------- (d) daily 500 hPa cut-off activity (ΣSo per cut-off day in the box) ----------
c500d = df500_all[(df500_all.ex == 1) & (df500_all.So >= 6.0)].copy()
daily = c500d.groupby("day")["So"].sum().to_numpy()               # ΣSo per day, over days that HAVE a cut-off
meanD = daily.mean()
bootD = rng.choice(daily, size=(N_BOOT, len(daily)), replace=True).mean(axis=1)
loD, hiD = np.percentile(bootD, [2.5, 97.5])
print(f"(d) daily activity: N={len(daily)} cut-off days  mean={meanD:.2f}  95%CI(mean)=[{loD:.2f},{hiD:.2f}]  median={np.median(daily):.2f}")
ax = axf[3]
ax.hist(daily, bins=np.arange(0, np.nanpercentile(daily, 99) + 3, 3.0), color="#2B7C8B", alpha=0.8, edgecolor="w")
ax.axvline(meanD, color="k", ls="--", lw=1.8, label=f"mean = {meanD:.1f}  (95% CI {loD:.1f}–{hiD:.1f})")
ax.axvspan(loD, hiD, color="k", alpha=0.18)
ax.set_xlabel("Daily Cut-Off Low activity  ΣSo  (So≥6, box)"); ax.set_ylabel("number of days")
ax.set_title(f"Daily 500 hPa cut-off activity, Cut-Off Low days (N={len(daily)})")
ax.legend(frameon=False); _letter(ax, "d")

plt.tight_layout(); plt.savefig("paper_figures/supplement_cutoff_detection.pdf"); plt.show()

In [ ]:
# ===== Is d4 log-normal? and candidate normalizations of cut-off activity =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

# daily SigmaSo (ex1, So>=6) over the full AMJJA calendar -> d4 = 5-day rolling sum
c = df500_all[(df500_all.ex == 1) & (df500_all.So >= 6.0)].copy()
daily = c.groupby("day")["So"].sum()
full = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
full = full[(full.month >= 4) & (full.month <= 8)]
daily = daily.reindex(full, fill_value=0.0)
d4 = daily.rolling(5, min_periods=1).sum()
active = d4[d4 > 0].to_numpy()
la = np.log(active)

gm = np.exp(la.mean())
print(f"d4 active days: N={len(active)}")
print(f"  arithmetic mean = {active.mean():.2f}   median = {np.median(active):.2f}   geometric mean = {gm:.2f}")
print(f"  skew(d4) = {stats.skew(active):+.2f}      skew(ln d4) = {stats.skew(la):+.2f}   (0 = symmetric)")
_sub = la if len(la) <= 5000 else np.random.default_rng(0).choice(la, 5000, replace=False)
print(f"  Shapiro normality of ln d4 (subsample): p = {stats.shapiro(_sub).pvalue:.2g}")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))
ax[0].hist(active, bins=45, color="#c0663a", alpha=0.85); ax[0].set_title(f"d4 (linear)  skew={stats.skew(active):+.2f}")
ax[0].set_xlabel("d4 = ΣSo (5-day)"); ax[0].axvline(active.mean(), color="k", ls="--", label="arith. mean")
ax[0].axvline(np.median(active), color="g", ls="--", label="median"); ax[0].axvline(gm, color="b", ls="--", label="geo. mean"); ax[0].legend(fontsize=9)
ax[1].hist(la, bins=45, color="#3a7fc0", alpha=0.85); ax[1].set_title(f"ln d4  skew={stats.skew(la):+.2f}"); ax[1].set_xlabel("ln d4")
stats.probplot(la, dist="norm", plot=ax[2]); ax[2].set_title("QQ: ln d4 vs Normal")
plt.tight_layout(); plt.show()

# ---- candidate normalized activity indices (on active days) ----
norm_mean = active / active.mean()            # current approach (fragile: heavy tail)
norm_med  = active / np.median(active)        # robust linear
mult_geo  = active / gm                        # multiplicative log-normal anomaly
z_log     = (la - la.mean()) / la.std()        # standardized log-anomaly (Gaussian-ish)
pctl      = stats.rankdata(active) / len(active)   # climatological percentile (distribution-free)
for lab, v in [("d4/mean", norm_mean), ("d4/median", norm_med), ("d4/geomean", mult_geo),
               ("z(log d4)", z_log), ("percentile", pctl)]:
    print(f"  {lab:12s}: skew={stats.skew(v):+.2f}  median={np.median(v):.2f}  IQR=[{np.percentile(v,25):.2f},{np.percentile(v,75):.2f}]")

In [ ]:
import numpy as np, pandas as pd
c = df500_all[(df500_all.ex == 1) & (df500_all.So >= 6.0)]
dailySo = c.groupby("day")["So"].sum()
full = pd.date_range(dailySo.index.min(), dailySo.index.max(), freq="D"); full = full[(full.month >= 4) & (full.month <= 8)]
allday = dailySo.reindex(full, fill_value=0.0); active = allday[allday > 0]
print(f"cut-off days: {len(active)} / {len(allday)} AMJJA days ({len(active)/len(allday):.1%})")
print(f"daily ΣSo | over cut-off days  : mean {active.mean():.2f}   median {active.median():.2f}   (skew {active.skew():+.2f})")
print(f"daily ΣSo | over ALL AMJJA days: mean {allday.mean():.2f}   median {allday.median():.2f}")
print(f"5-day d4 baseline (daily mean x5): cut-off days {active.mean()*5:.1f}   |   all-days {allday.mean()*5:.1f}")

In [ ]:
import pandas as pd
mm = pd.DatetimeIndex(df500_all["day"]).month
print("months present:", sorted(mm.unique()), "| year range:", df500_all["day"].dt.year.min(), "-", df500_all["day"].dt.year.max())

In [ ]:
# ===== z500(So>=6) cut-offs with NO z200 cut-off above =====
# Reuses df200_all, df500_all, _hav.  A z500 low is "capped" if a z200 cut-off lies within
# the z200 feature's So-optimal radius on the same day.
import numpy as np, pandas as pd
MATCH_RADIUS_KM, RADIUS_FACTOR, FALLBACK_RADIUS_KM = None, 1.0, 800.0

c500q   = df500_all[(df500_all.ex == 1) & (df500_all.So >= 6.0)].copy()
c200any = df200_all[df200_all.ex == 1].copy()                        # z200 at ANY So (>= floor ~3)
c200_10 = df200_all[(df200_all.ex == 1) & (df200_all.So >= 10.0)].copy()

def has_partner(src, tgt):
    """set of src indices with >=1 tgt within tgt's own radius, same day."""
    tby = {d: (g["lat"].to_numpy(), g["lon"].to_numpy(), g["R"].to_numpy()) for d, g in tgt.groupby("day")}
    hit = set()
    for d, gs in src.groupby("day"):
        t = tby.get(d)
        if t is None:
            continue
        latt, lont, Rt = t
        for i, r in gs.iterrows():
            dist = _hav(r["lat"], r["lon"], latt, lont); j = int(np.argmin(dist))
            tol = MATCH_RADIUS_KM if MATCH_RADIUS_KM is not None else \
                  (Rt[j] * RADIUS_FACTOR if np.isfinite(Rt[j]) else FALLBACK_RADIUS_KM)
            if dist[j] <= tol:
                hit.add(i)
    return hit

n = len(c500q)
print(f"z500 cut-offs (So>=6): N={n}")
for lab, tgt in [("z200 (any So)", c200any), ("z200 (So>=10)", c200_10)]:
    w = has_partner(c500q, tgt); nw = len(w); no = n - nw
    print(f"  with {lab:14s} above: {nw:5d} ({nw/n:.1%})  |  WITHOUT: {no:5d} ({no/n:.1%})")

In [ ]:
# ===== Do orphan z500(So>=6) cut-offs (no SAME-DAY z200) have a z200 cut-off the DAY BEFORE? =====
# Reuses df200_all, df500_all, _hav.  NB: lagged search uses the z500 low's day-D position; because a
# cut-off propagates ~500 km/day, a wider fixed radius is fairer for the +/-1 day search (see LAG_RADIUS_KM).
import numpy as np, pandas as pd
SAMEDAY_RADIUS = None       # None -> z200 feature's own So-radius (same-day matching)
LAG_RADIUS_KM  = 1000.0     # fixed radius for the D-1 / D+1 search (allows ~1 day of system motion)
RADIUS_FACTOR, FALLBACK_RADIUS_KM = 1.0, 800.0

c500q   = df500_all[(df500_all.ex == 1) & (df500_all.So >= 6.0)].copy()
c200any = df200_all[df200_all.ex == 1].copy()
c200_10 = df200_all[(df200_all.ex == 1) & (df200_all.So >= 10.0)].copy()

def has_partner_on(src, tgt, day_offset=0, radius_km=None):
    tby = {d: (g["lat"].to_numpy(), g["lon"].to_numpy(), g["R"].to_numpy()) for d, g in tgt.groupby("day")}
    hit = set()
    for d, gs in src.groupby("day"):
        t = tby.get(d + pd.Timedelta(days=day_offset))
        if t is None:
            continue
        latt, lont, Rt = t
        for i, r in gs.iterrows():
            dist = _hav(r["lat"], r["lon"], latt, lont); j = int(np.argmin(dist))
            tol = radius_km if radius_km is not None else \
                  (Rt[j] * RADIUS_FACTOR if np.isfinite(Rt[j]) else FALLBACK_RADIUS_KM)
            if dist[j] <= tol:
                hit.add(i)
    return hit

# same-day orphans: no z200 of ANY So above on day D
same_any = has_partner_on(c500q, c200any, 0, SAMEDAY_RADIUS)
orphans = c500q[~c500q.index.isin(same_any)].copy(); No = len(orphans)
print(f"orphan z500(So>=6), NO same-day z200 (any So): N={No} ({No/len(c500q):.1%} of {len(c500q)})")
print(f"[lagged search radius = {LAG_RADIUS_KM:.0f} km]")
for lab, tgt in [("z200 any So", c200any), ("z200 So>=10", c200_10)]:
    dm1 = has_partner_on(orphans, tgt, -1, LAG_RADIUS_KM)
    dp1 = has_partner_on(orphans, tgt, +1, LAG_RADIUS_KM)
    print(f"  {lab:11s} on D-1: {len(dm1):5d} ({len(dm1)/No:.1%})   on D+1: {len(dp1):5d} ({len(dp1)/No:.1%})")

dm1a = has_partner_on(orphans, c200any, -1, LAG_RADIUS_KM)
dp1a = has_partner_on(orphans, c200any, +1, LAG_RADIUS_KM)
either = dm1a | dp1a
print(f"  z200(any) D-1 only (decaying, not D+1): {len(dm1a - dp1a):5d} ({len(dm1a - dp1a)/No:.1%})")
print(f"  z200(any) D+1 only (forming, not D-1):  {len(dp1a - dm1a):5d} ({len(dp1a - dm1a)/No:.1%})")
print(f"  z200(any) on D-1 OR D+1:                {len(either):5d} ({len(either)/No:.1%})")
print(f"  z200(any) on NEITHER D-1,D,D+1:         {No - len(either):5d} ({(No - len(either))/No:.1%})")

In [ ]:
# ============================================================================
# So temporal-averaging calibration at the STUDY grid (1x1 deg), from the ERA5(T)
# pool GRIB. CDO remapbil -> 1deg hourly; detect peak So of the Iberian cut-off
# on instantaneous / 6h / 12h / daily-mean fields. FULL detection region.
# ============================================================================
import numpy as np, pandas as pd, xarray as xr, glob, os
from pathlib import Path
from cdo import Cdo
from colindex2 import Detect

os.environ["PATH"] = "/path/to/cdo/bin" + os.pathsep + os.environ.get("PATH", "")
ARCH = "/pool/data/ERA5/ET/pl/an/1H/129"        # 2026 ERA5T hourly z500 (param 129)
EVENT_DAY = "2026-05-25"; REF = pd.Timestamp(EVENT_DAY + " 12:00")
DET_LON, DET_LAT = (-50, 55), (5, 90)           # FULL detection region
COL_LON, COL_LAT = (-35, 0), (25, 50)           # Iberian box (peak So read here)
RADII = np.arange(200, 2101, 100); SO_DETECT, SR_MAX = 3.0, 3.0
WORK = Path("./so_calib"); WORK.mkdir(exist_ok=True)

# --- 1deg target grid (+1deg buffer) ---
b = 1.0
xn = int(round((DET_LON[1] - DET_LON[0] + 2 * b) / 1.0)) + 1
yn = int(round((DET_LAT[1] - DET_LAT[0] + 2 * b) / 1.0)) + 1
GRID = WORK / "grid_1deg.txt"
GRID.write_text("\n".join(["gridtype = lonlat", f"xsize = {xn}", f"ysize = {yn}",
                           f"xfirst = {DET_LON[0]-b}", "xinc = 1.0",
                           f"yfirst = {DET_LAT[0]-b}", "yinc = 1.0", ""]))

# --- CDO remapbil native hourly -> 1deg hourly (no daymean; keep sub-daily) ---
files = sorted({f for pat in (f"*{EVENT_DAY}*.grb", f"*{EVENT_DAY.replace('-','')}*.grb")
                for f in glob.glob(os.path.join(ARCH, pat)) if os.path.isfile(f)})
if not files:
    raise FileNotFoundError(f"No z500 GRIBs for {EVENT_DAY} under {ARCH}")
print(f"found {len(files)} pool GRIB(s)")
nc1 = str(WORK / "z500_1deg_day.nc")
Cdo().remapbil(str(GRID),
               input=f"-sellonlatbox,{DET_LON[0]-b},{DET_LON[1]+b},{DET_LAT[0]-b},{DET_LAT[1]+b} "
                     f"-sellevel,50000 -setgridtype,regular -mergetime {' '.join(files)}",
               output=nc1, options="-O -f nc4 --eccodes")

_src = xr.open_dataarray(nc1)
for d in list(_src.dims):
    if d not in ("time", "lat", "lon"): _src = _src.isel({d: 0}, drop=True)
day = _src.assign_coords(lon=(((_src["lon"] + 180) % 360) - 180)).sortby("lon").sortby("lat").load()
if float(day.mean()) > 1e4:
    day = day / 9.80665
_src.close()
print(f"loaded 1deg day: {dict(day.sizes)}")

def _peak_So(f2d, tag):
    f2d = f2d.drop_vars([c for c in ("time", "level") if c in f2d.coords], errors="ignore")
    t2 = pd.to_datetime([REF, REF + pd.Timedelta("1D")])          # 2 dummy steps (dodge single-timestep bug)
    h = xr.concat([f2d, f2d], dim="time").assign_coords(time=t2, level=500)
    h = h.assign_attrs(units="m", standard_name="geopotential_height").transpose("time", "lat", "lon")
    outdir = WORK / f"det_{tag}"; outdir.mkdir(exist_ok=True); prev = os.getcwd(); os.chdir(outdir)
    try:
        Detect(h, odir="out", ty="L", r=RADII, SR_thres=SR_MAX, So_thres=SO_DETECT, Do_thres=0.0,
               xx_thres=0.0, rm_rmin=True, rm_rmax=True, local_ex_req_num=7, gpu=False, chunk_size=100000, nc=True, subs=False)
    finally:
        os.chdir(prev)
    fr = [pd.read_csv(f, parse_dates=["time"]) for f in glob.glob(str(outdir / "out" / "V" / "*" / "V-L-*.csv"))]
    pts = pd.concat([f for f in fr if not f.empty], ignore_index=True) if fr else pd.DataFrame(columns=["lon", "lat", "So", "ex"])
    if pts.empty: return np.nan
    pts["ex"] = pd.to_numeric(pts["ex"], errors="coerce"); lon = ((pts["lon"] + 180) % 360) - 180
    q = pts[(pts["ex"] == 1) & lon.between(*COL_LON) & pts["lat"].between(*COL_LAT)]
    return float(q["So"].max()) if len(q) else np.nan

print("\nTEMPORAL sweep (1x1 deg):")
tp = {}
tp["instantaneous"] = _peak_So(day.sel(time=REF, method="nearest"), "t_inst")
tp["6h mean"]  = _peak_So(day.sel(time=slice(EVENT_DAY + " 09:00", EVENT_DAY + " 14:00")).mean("time"), "t_6h")
tp["12h mean"] = _peak_So(day.sel(time=slice(EVENT_DAY + " 06:00", EVENT_DAY + " 17:00")).mean("time"), "t_12h")
tp["daily mean"] = _peak_So(day.mean("time"), "t_24h")
for k, v in tp.items():
    print(f"  {k:>13} : peak So = {v:.2f} m/100km")
if np.isfinite(tp["instantaneous"]) and np.isfinite(tp["daily mean"]):
    print(f"\n1deg: instantaneous {tp['instantaneous']:.1f} -> daily-mean {tp['daily mean']:.1f}  "
          f"(temporal ratio {tp['daily mean']/tp['instantaneous']:.2f})")

In [ ]:
# ============================================================================
# Same 1deg So temporal calibration, on the "verge" analogue 2025-06-17
# (flipped factual->counterfactual between So>=6 and So>=10).
# ERA5 (E5) archive for pre-2026 dates; ERA5T (ET) for 2026.
# ============================================================================
import numpy as np, pandas as pd, xarray as xr, glob, os
from pathlib import Path
from cdo import Cdo
from colindex2 import Detect

os.environ["PATH"] = "/path/to/cdo/bin" + os.pathsep + os.environ.get("PATH", "")
EVENT_DAY = "2025-06-17"                          # <-- the verge analogue (change freely)
REF = pd.Timestamp(EVENT_DAY + " 12:00")
YEAR = int(EVENT_DAY[:4])
ARCH = f"/pool/data/ERA5/{'ET' if YEAR >= 2026 else 'E5'}/pl/an/1H/129"   # ERA5T 2026, else ERA5 hourly
DET_LON, DET_LAT = (-50, 55), (5, 90)
COL_LON, COL_LAT = (-35, 0), (25, 50)
RADII = np.arange(200, 2101, 100); SO_DETECT, SR_MAX = 3.0, 3.0
WORK = Path("./so_calib"); WORK.mkdir(exist_ok=True)

b = 1.0
xn = int(round((DET_LON[1] - DET_LON[0] + 2 * b) / 1.0)) + 1
yn = int(round((DET_LAT[1] - DET_LAT[0] + 2 * b) / 1.0)) + 1
GRID = WORK / "grid_1deg.txt"
GRID.write_text("\n".join(["gridtype = lonlat", f"xsize = {xn}", f"ysize = {yn}",
                           f"xfirst = {DET_LON[0]-b}", "xinc = 1.0",
                           f"yfirst = {DET_LAT[0]-b}", "yinc = 1.0", ""]))

files = sorted({f for pat in (f"*{EVENT_DAY}*.grb", f"*{EVENT_DAY.replace('-','')}*.grb")
                for f in glob.glob(os.path.join(ARCH, pat)) if os.path.isfile(f)})
if not files:
    raise FileNotFoundError(f"No z500 GRIBs for {EVENT_DAY} under {ARCH}")
print(f"found {len(files)} pool GRIB(s) under {ARCH}")
nc1 = str(WORK / f"z500_1deg_{EVENT_DAY}.nc")
Cdo().remapbil(str(GRID),
               input=f"-sellonlatbox,{DET_LON[0]-b},{DET_LON[1]+b},{DET_LAT[0]-b},{DET_LAT[1]+b} "
                     f"-sellevel,50000 -setgridtype,regular -mergetime {' '.join(files)}",
               output=nc1, options="-O -f nc4 --eccodes")

_src = xr.open_dataarray(nc1)
for d in list(_src.dims):
    if d not in ("time", "lat", "lon"): _src = _src.isel({d: 0}, drop=True)
day = _src.assign_coords(lon=(((_src["lon"] + 180) % 360) - 180)).sortby("lon").sortby("lat").load()
if float(day.mean()) > 1e4:
    day = day / 9.80665
_src.close()
print(f"loaded 1deg day: {dict(day.sizes)}")

def _peak_So(f2d, tag):
    f2d = f2d.drop_vars([c for c in ("time", "level") if c in f2d.coords], errors="ignore")
    t2 = pd.to_datetime([REF, REF + pd.Timedelta("1D")])
    h = xr.concat([f2d, f2d], dim="time").assign_coords(time=t2, level=500)
    h = h.assign_attrs(units="m", standard_name="geopotential_height").transpose("time", "lat", "lon")
    outdir = WORK / f"det_{EVENT_DAY}_{tag}"; outdir.mkdir(exist_ok=True); prev = os.getcwd(); os.chdir(outdir)
    try:
        Detect(h, odir="out", ty="L", r=RADII, SR_thres=SR_MAX, So_thres=SO_DETECT, Do_thres=0.0,
               xx_thres=0.0, rm_rmin=True, rm_rmax=True, local_ex_req_num=7, gpu=False, chunk_size=100000, nc=True, subs=False)
    finally:
        os.chdir(prev)
    fr = [pd.read_csv(f, parse_dates=["time"]) for f in glob.glob(str(outdir / "out" / "V" / "*" / "V-L-*.csv"))]
    pts = pd.concat([f for f in fr if not f.empty], ignore_index=True) if fr else pd.DataFrame(columns=["lon", "lat", "So", "ex"])
    if pts.empty: return np.nan
    pts["ex"] = pd.to_numeric(pts["ex"], errors="coerce"); lon = ((pts["lon"] + 180) % 360) - 180
    q = pts[(pts["ex"] == 1) & lon.between(*COL_LON) & pts["lat"].between(*COL_LAT)]
    return float(q["So"].max()) if len(q) else np.nan

print(f"\nTEMPORAL sweep (1x1 deg), {EVENT_DAY}:")
tp = {}
tp["instantaneous"] = _peak_So(day.sel(time=REF, method="nearest"), "inst")
tp["6h mean"]  = _peak_So(day.sel(time=slice(EVENT_DAY + " 09:00", EVENT_DAY + " 14:00")).mean("time"), "6h")
tp["12h mean"] = _peak_So(day.sel(time=slice(EVENT_DAY + " 06:00", EVENT_DAY + " 17:00")).mean("time"), "12h")
tp["daily mean"] = _peak_So(day.mean("time"), "24h")
for k, v in tp.items():
    print(f"  {k:>13} : peak So = {v:.2f} m/100km")
if np.isfinite(tp["instantaneous"]) and np.isfinite(tp["daily mean"]):
    print(f"\n{EVENT_DAY} 1deg: instantaneous {tp['instantaneous']:.1f} -> daily-mean {tp['daily mean']:.1f}  "
          f"(ratio {tp['daily mean']/tp['instantaneous']:.2f})  | Kasuga So>10? "
          f"inst {'YES' if tp['instantaneous']>10 else 'no'}, daily {'YES' if tp['daily mean']>10 else 'no'}")

In [ ]:
# ============================================================================
# So at 300 hPa vs 500 hPa (peak of the Iberian cut-off), 1x1 deg, for the event
# (2026-05-25) and the verge analogue (2025-06-17). instantaneous + daily-mean.
# ============================================================================
import numpy as np, pandas as pd, xarray as xr, glob, os
from pathlib import Path
from cdo import Cdo
from colindex2 import Detect

os.environ["PATH"] = "/path/to/cdo/bin" + os.pathsep + os.environ.get("PATH", "")
DAYS = ["2026-05-25", "2025-06-17"]
LEVELS = {"300hPa": 30000, "500hPa": 50000}       # param 129 geopotential at these pressures (Pa)
DET_LON, DET_LAT = (-50, 55), (5, 90)
COL_LON, COL_LAT = (-35, 0), (25, 50)
RADII = np.arange(200, 2101, 100); SO_DETECT, SR_MAX = 3.0, 3.0
WORK = Path("./so_calib"); WORK.mkdir(exist_ok=True)

b = 1.0
xn = int(round((DET_LON[1] - DET_LON[0] + 2 * b) / 1.0)) + 1
yn = int(round((DET_LAT[1] - DET_LAT[0] + 2 * b) / 1.0)) + 1
GRID = WORK / "grid_1deg.txt"
GRID.write_text("\n".join(["gridtype = lonlat", f"xsize = {xn}", f"ysize = {yn}",
                           f"xfirst = {DET_LON[0]-b}", "xinc = 1.0",
                           f"yfirst = {DET_LAT[0]-b}", "yinc = 1.0", ""]))

def _peak_So(f2d, tag):
    f2d = f2d.drop_vars([c for c in ("time", "level") if c in f2d.coords], errors="ignore")
    t2 = pd.to_datetime(["2000-01-01", "2000-01-02"])
    h = xr.concat([f2d, f2d], dim="time").assign_coords(time=t2, level=500)
    h = h.assign_attrs(units="m", standard_name="geopotential_height").transpose("time", "lat", "lon")
    outdir = WORK / f"det_{tag}"; outdir.mkdir(exist_ok=True); prev = os.getcwd(); os.chdir(outdir)
    try:
        Detect(h, odir="out", ty="L", r=RADII, SR_thres=SR_MAX, So_thres=SO_DETECT, Do_thres=0.0,
               xx_thres=0.0, rm_rmin=True, rm_rmax=True, local_ex_req_num=7, gpu=False, chunk_size=100000, nc=True, subs=False)
    finally:
        os.chdir(prev)
    fr = [pd.read_csv(f, parse_dates=["time"]) for f in glob.glob(str(outdir / "out" / "V" / "*" / "V-L-*.csv"))]
    pts = pd.concat([f for f in fr if not f.empty], ignore_index=True) if fr else pd.DataFrame(columns=["lon", "lat", "So", "ex"])
    if pts.empty: return np.nan
    pts["ex"] = pd.to_numeric(pts["ex"], errors="coerce"); lon = ((pts["lon"] + 180) % 360) - 180
    q = pts[(pts["ex"] == 1) & lon.between(*COL_LON) & pts["lat"].between(*COL_LAT)]
    return float(q["So"].max()) if len(q) else np.nan

def calib(day_str, level_pa, tag):
    yr = int(day_str[:4]); arch = f"/pool/data/ERA5/{'ET' if yr >= 2026 else 'E5'}/pl/an/1H/129"
    files = sorted({f for pat in (f"*{day_str}*.grb", f"*{day_str.replace('-','')}*.grb")
                    for f in glob.glob(os.path.join(arch, pat)) if os.path.isfile(f)})
    if not files:
        raise FileNotFoundError(f"No GRIBs for {day_str} under {arch}")
    nc = str(WORK / f"z_{level_pa}_{day_str}.nc")
    Cdo().remapbil(str(GRID),
                   input=f"-sellonlatbox,{DET_LON[0]-b},{DET_LON[1]+b},{DET_LAT[0]-b},{DET_LAT[1]+b} "
                         f"-sellevel,{level_pa} -setgridtype,regular -mergetime {' '.join(files)}",
                   output=nc, options="-O -f nc4 --eccodes")
    s = xr.open_dataarray(nc)
    for d in list(s.dims):
        if d not in ("time", "lat", "lon"): s = s.isel({d: 0}, drop=True)
    fld = s.assign_coords(lon=(((s["lon"] + 180) % 360) - 180)).sortby("lon").sortby("lat").load()
    if float(fld.mean()) > 1e4: fld = fld / 9.80665
    s.close()
    ref = pd.Timestamp(day_str + " 12:00")
    return (_peak_So(fld.sel(time=ref, method="nearest"), tag + "_inst"),
            _peak_So(fld.mean("time"), tag + "_daily"))

res = {}
print(f"{'day':>12} {'level':>7} | {'So inst':>8} {'So daily':>9}")
for day_str in DAYS:
    for lname, lpa in LEVELS.items():
        si, sd = calib(day_str, lpa, f"{day_str}_{lname}")
        res[(day_str, lname)] = (si, sd)
        print(f"{day_str:>12} {lname:>7} | {si:8.2f} {sd:9.2f}")

print("\n300hPa / 500hPa scaling (daily-mean, 1deg):")
for day_str in DAYS:
    s3 = res[(day_str, "300hPa")][1]; s5 = res[(day_str, "500hPa")][1]
    if np.isfinite(s3) and np.isfinite(s5) and s5:
        print(f"  {day_str}: So300={s3:.1f}  So500={s5:.1f}  ratio {s3/s5:.2f}"
              f"  | >10 at 300 {'YES' if s3>10 else 'no'}, at 500 {'YES' if s5>10 else 'no'}")